# Lesson 04 — Motion Blur and Deconvolution

## Why This Lesson
Motion blur happens when the camera or subject moves during exposure.
Deconvolution tries to reverse it — mathematically "undoing" the blur.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img  = cv2.imread('sample.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0

# Simulate horizontal motion blur
def motion_blur_kernel(size, angle=0):
    kernel = np.zeros((size, size))
    kernel[size//2, :] = 1.0 / size
    M = cv2.getRotationMatrix2D((size//2, size//2), angle, 1)
    return cv2.warpAffine(kernel, M, (size, size))

k_horizontal = motion_blur_kernel(25, 0)
k_diagonal   = motion_blur_kernel(25, 45)

blurred_h = cv2.filter2D(gray, -1, k_horizontal)
blurred_d = cv2.filter2D(gray, -1, k_diagonal)

# Wiener deconvolution (frequency domain)
def wiener_deconv(blurred, kernel, noise_var=0.01):
    h, w = blurred.shape
    kh, kw = kernel.shape
    kernel_padded = np.zeros((h, w))
    kernel_padded[:kh, :kw] = kernel
    K  = np.fft.fft2(kernel_padded)
    B  = np.fft.fft2(blurred)
    # Wiener filter: H* / (|H|^2 + noise)
    K_conj = np.conj(K)
    W = K_conj / (np.abs(K)**2 + noise_var)
    restored = np.real(np.fft.ifft2(W * B))
    return np.clip(restored, 0, 1)

restored = wiener_deconv(blurred_h, k_horizontal, noise_var=0.005)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, im, t in zip(axes,
    [gray, blurred_h, restored],
    ['Original', 'Motion blurred (horizontal)', 'Wiener deconvolution']):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.suptitle('Deconvolution works when you know the blur kernel — imperfect but useful', fontsize=12)
plt.show()

## Key Takeaway
Deconvolution only works well when you know the exact blur kernel.
In real blind deconvolution (unknown kernel), it's much harder.
The noise parameter `noise_var` controls the tradeoff between sharpness and ringing artifacts.